In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv(r"C:\Users\joseph.persteins\Downloads\USE THIS FOR NLP 2.csv")

# Inspect the first few rows of the dataset
df.head()

,drug_name_x,condition,review,rating,date,Ingredient,Approval_Date,Type,Applicant_Full_Name,Best Match Medicine Name,drug_name_y,Extracted Information
0,abilify,autism,"""my child has been on abilify for a while now ...",1,28-May-17,aripiprazole,20-Sep-06,discn,otsuka pharmaceutical co ltd,AIR,AIR,bupropion hydrochloride extended-release tabl...
1,abraxane,breast cance,"""this medicine saved my life! i received it we...",10,28-Oct-12,paclitaxel,7-Jan-05,rx,bristol-myers squibb co,PROPARACAINE HYDROCHLORIDE OPHTHALMIC SOLUTION...,PROPARACAINE HYDROCHLORIDE OPHTHALMIC SOLUTION...,"hypertension metoprolol tartrate tablets, usp..."
2,abreva,herpes simplex,"""i woke up 3 days ago with a huge painful outb...",10,4-Jan-16,docosanol,25-Jul-00,otc,haleon us holdings llc,RENOVA®,RENOVA®,1 detrol la capsules is indicated for the tr...
3,absorica,acne,"""i&rsquo;m a 23 year old female who has had ac...",10,24-Jun-14,isotretinoin,25-May-12,rx,sun pharmaceutical industries inc,S,S,carisoprodol is indicated for the relief of d...
4,acanya,acne,"""thank god i read the reviews before applying ...",1,17-Dec-15,benzoyl peroxide; clindamycin phosphate,23-Oct-08,rx,bausch health americas inc,"AMANTADINE HYDROCHLORIDE CAPSULES, USP","AMANTADINE HYDROCHLORIDE CAPSULES, USP",phenazopyridine hcl is indicated for the symp...


In [3]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

In [4]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def nltk_clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)  # Convert to string
    # Remove user @ references and '#' from tweet
    text = re.sub(r'\@\w+|\#', '', str(text))  # Convert to string
    # Tokenize text
    tokens = word_tokenize(text)
    # Convert to lower case
    tokens = [w.lower() for w in tokens]
    # Remove punctuation from each word
    words = [word for word in tokens if word.isalpha()]
    # Filter out stop words
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if not w in stop_words]
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\joseph.persteins\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\joseph.persteins\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\joseph.persteins\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
def categorize_rating(rating):
    rating = int(rating)  # Convert rating to integer
    if rating > 6:
        return 'positive'
    elif rating == 6:
        return 'neutral'
    else:
        return 'negative'

In [6]:
# Preprocessing steps (nltk_clean_text, categorize_rating) remain the same

# Assuming 'df' is your DataFrame containing the data
df['rating_category'] = df['rating'].apply(categorize_rating)

# Apply the improved cleaning function
df['review'] = df['review'].apply(nltk_clean_text)
df['Extracted Information'] = df['Extracted Information'].apply(nltk_clean_text)
# Assuming 'condition' column exists and needs preprocessing similar to 'review'
df['condition'] = df['condition'].apply(nltk_clean_text)
df['drug_name_x'] = df['drug_name_x'].apply(nltk_clean_text)

# Combine text columns and include 'condition'
df['combined_text'] = df['review'] + " " + df['drug_name_x'] + " " + df['rating_category']

In [15]:
max_len = max(len(text) for text in df['combined_text'])
print(max_len)

939


In [8]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification

# Encode the labels
label_encoder = LabelEncoder()
df['rating_category_encoded'] = label_encoder.fit_transform(df['rating_category'])
labels = to_categorical(df['rating_category_encoded'], num_classes=3)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(df['combined_text'], labels, test_size=0.2, random_state=42)

# Initialize tokenizer and encode reviews
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

def encode_reviews(reviews):
    return tokenizer.batch_encode_plus(reviews.to_list(), padding=True, truncation=True, max_length=512, return_tensors='tf')
X_train_enc = encode_reviews(X_train)
X_test_enc = encode_reviews(X_test)


# Compile model with optimizer and loss function
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Convert input data to tensors
train_dataset = tf.data.Dataset.from_tensor_slices(({"input_ids": X_train_enc['input_ids'], "attention_mask": X_train_enc['attention_mask']}, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices(({"input_ids": X_test_enc['input_ids'], "attention_mask": X_test_enc['attention_mask']}, y_test))

# Train the model with early stopping
#early_stopping_cb = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
model.fit(train_dataset.shuffle(len(X_train)).batch(8), epochs=10, batch_size=8)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

C:\Users\joseph.persteins\AppData\Local\anaconda31\Lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joseph.persteins\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 


Epoch 1/10
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert



164/167 [============================>.] - ETA: 7s - loss: 11.3388 - accuracy: 0.0892 


KeyboardInterrupt

